<a href="https://colab.research.google.com/github/shobha-nosimpler/GenAI/blob/main/GenAi_embeddings_question_answer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# What are embeddings?
Embeddings are dense vector representations of data (e.g., words, sentences, or documents) that capture the semantic meaning of the data in a continuous vector space. These vectors allow machine learning models to understand and manipulate data more effectively, especially in tasks involving natural language processing (NLP).

# Use Case

Embedding-Based Search for Knowledge Retrieval in Dialogue Applications

We create a knowledge base of questions and their corresponding answer
- the most commonly asked questions are listed and then embedding using the openai embeddings model
- a query is then mapped to the closest resembling query in the knowledge base and the corresponding answere is returned

# About the use case
Let's implement a simple end-to-end use case where an embedding-based search is used to retrieve relevant information in a long conversation, particularly in a banking scenario. The goal is to create a chatbot that can answer banking-related queries by retrieving the most relevant responses from a pre-defined knowledge base.

In [2]:
#!pip install openai
!pip install openai==0.28

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 3.1 MB/s eta 0:00:00


In [3]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 32.1 MB/s eta 0:00:00


In [4]:
import openai
import faiss
import numpy as np

In [ ]:
# Initialize the OpenAI API with your key
#openai.api_key = 'your-api-key'
# pass secret key for authentication
api_key = input("Enter your API key: ")
openai.api_key = api_key

In [1]:
# Sample knowledge base questions and answers
knowledge_base = [
    {"question": "How can I open a new bank account?", "answer": "You can open a new bank account by visiting your nearest branch or applying online on our website."},
    {"question": "What are the interest rates for savings accounts?", "answer": "The interest rates for savings accounts vary; please check our website or visit your nearest branch for current rates."},
    {"question": "How can I apply for a credit card?", "answer": "You can apply for a credit card online or at any of our branches."},
    {"question": "What is the procedure to report a lost card?", "answer": "To report a lost card, call our customer service immediately and follow the instructions."},
    {"question": "How can I check my account balance?", "answer": "You can check your account balance via online banking, our mobile app, or at any ATM."},
    {"question": "What are the current loan options available?", "answer": "We offer various loan options including personal, home, and auto loans; please visit our website for more details."},
    {"question": "How can I update my contact information?", "answer": "You can update your contact information through online banking or by visiting a branch."},
    {"question": "What are the bank's operating hours?", "answer": "Our bank's operating hours are 9 AM to 5 PM from Monday to Friday."},
]



In [6]:
# Separate questions and answers
questions = [item["question"] for item in knowledge_base]
answers = [item["answer"] for item in knowledge_base]



In [7]:
# Encode the knowledge base questions using OpenAI's embedding model
def get_embeddings(text_list):
    response = openai.Embedding.create(
        model="text-embedding-ada-002",
        input=text_list
    )
    embeddings = [record['embedding'] for record in response['data']]
    return np.array(embeddings)



In [8]:
question_embeddings = get_embeddings(questions)

In [9]:
print(question_embeddings.shape)

(8, 1536)


In [10]:
# Create FAISS index
d = question_embeddings.shape[1]  # Dimension of the embeddings
index = faiss.IndexFlatL2(d)
index.add(question_embeddings)

# Define the Search Function

In [13]:
def search_knowledge_base(query):
    # Get the embedding for the query
    query_embedding = get_embeddings([query])

    # Search the FAISS index for the nearest neighbors
    D, I = index.search(query_embedding, k=1)

    # Return the most relevant response
    return (questions[I[0][0]],answers[I[0][0]])


# Modify the Search Function that returns a match upto the minimum desired threshold

In [26]:
def search_knowledge_base_threshold(query, threshold=0.6):
    # Get the embedding for the query
    query_embedding = get_embeddings([query])

    # Search the FAISS index for the nearest neighbors
    D, I = index.search(query_embedding, k=1)

    # Check if the best match is within the threshold
    if D[0][0] < threshold:
        return (questions[I[0][0]],answers[I[0][0]])
    else:
        return "I'm sorry, I don't have the information you're looking for. Can you please rephrase your question or ask something else?"


In [14]:
# Example usage
query = "How can I get a new debit card?"
#response = search_knowledge_base(query)
question, answer = search_knowledge_base(query)
print("Query:", question)
print("Response:", answer)

Query: How can I apply for a credit card?
Response: You can apply for a credit card online or at any of our branches.


In [15]:
# Example usage
query2 = "How can I update my contact information?"
#response = search_knowledge_base(query)
question2, answer2 = search_knowledge_base(query2)
print("Query:", question2)
print("Response:", answer2)

Query: How can I update my contact information?
Response: You can update your contact information through online banking or by visiting a branch.


# Implement Chatbot with Context Memory

In [34]:
class Chatbot:
    def __init__(self):
        self.history = []

    def ask(self, query):
        self.history.append(query)
        context = " ".join(self.history)
        question, answer = search_knowledge_base_threshold(context,1)
        response = (question, answer)
        self.history.append(response)
        return response

In [18]:
# Example usage
# Note: query is mapped to the most closest match query in the knowledge base
chatbot = Chatbot()
#print(chatbot.ask("What are your operating hours?"))
print(chatbot.ask("Can you tell me about the interest rates for savings accounts?"))

('What are the interest rates for savings accounts?', 'The interest rates for savings accounts vary; please check our website or visit your nearest branch for current rates.')


In [24]:
#Creat a new instance for the new query
chatbot = Chatbot()
print(chatbot.ask("What are your operating hours?"))

("What are the bank's operating hours?", "Our bank's operating hours are 9 AM to 5 PM from Monday to Friday.")


In [35]:
#Creat a new instance for the new query
chatbot = Chatbot()
print(chatbot.ask("How many planets are there?"))

('How can I check my account balance?', 'You can check your account balance via online banking, our mobile app, or at any ATM.')


In [36]:
#Creat a new instance for the new query
chatbot = Chatbot()
print(chatbot.ask("?"))

('How can I open a new bank account?', 'You can open a new bank account by visiting your nearest branch or applying online on our website.')
